# 🎙️ 日本語ウェイクワード「先生」学習ノートブック

このノートブックは **livekit-wakeword-ja** フォークを使って、日本語ウェイクワード「先生（せんせい）」の ONNX モデルを学習する手順を示します。

---

## 概要

1. [前提確認（GPU・環境）](#gpu)
2. [システム依存パッケージのインストール](#sys)
3. [リポジトリのクローンとパッケージインストール](#install)
4. [設定ファイルの確認](#config)
5. [セットアップ（モデル・データのダウンロード）](#setup)
6. [合成データ生成](#generate)
7. [オーグメンテーション](#augment)
8. [学習](#train)
9. [ONNX エクスポート](#export)
10. [評価](#eval)
11. [モデルのダウンロード](#download)
12. [トラブルシューティング](#troubleshoot)

---

### 推奨ランタイム

- **GPU: T4 以上**（「ランタイム」→「ランタイムのタイプを変更」→ T4 GPU）
- Colab Pro / Pro+ を推奨（学習に 1〜4 時間かかる場合あり）

> **⚠️ TODO**: VoxCPM2 の日本語合成品質はエンドツーエンドで未検証です。合成クリップを確認しながら進めてください。

## 1. 前提確認（GPU・環境）

In [ ]:
# GPU が割り当てられているか確認
!nvidia-smi
import sys
print(f"Python: {sys.version}")

# GPU が表示されない場合は「ランタイム」→「ランタイムのタイプを変更」→ T4 GPU に変更してください

## 2. システム依存パッケージのインストール

`espeak-ng` はアドバーサリアルフレーズの音素分解に必要です。

In [ ]:
!apt-get install -y -q espeak-ng
!espeak-ng --version

## 3. リポジトリのクローンとパッケージインストール

> フォーク URL は実際の GitHub URL に書き換えてください。

In [ ]:
import os

REPO_URL = "https://github.com/hazukit/livekit-wakeword-ja.git"  # 必要に応じて変更
REPO_DIR = "/content/livekit-wakeword-ja"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} は既にクローン済みです")

os.chdir(REPO_DIR)
!pwd

In [ ]:
# train と voxcpm extras をインストール
!pip install -q -e ".[train,voxcpm]"

# インストール確認
!livekit-wakeword --help

## 4. 設定ファイルの確認

`examples/sensei_ja.yaml` を確認します。主要な変更ポイント:

- `model_name`: 出力ディレクトリとエクスポートファイル名
- `target_phrases`: ウェイクワード（漢字・ひらがな両形式）
- `custom_negative_phrases`: 手動アドバーサリアルフレーズ
- `n_samples`: 合成クリップ数（デフォルト 25,000）
- `steps`: 学習ステップ数（デフォルト 100,000）

In [ ]:
CONFIG = "examples/sensei_ja.yaml"

with open(CONFIG) as f:
    print(f.read())

In [ ]:
# YAML 構文の検証
import yaml
with open(CONFIG) as f:
    cfg = yaml.safe_load(f)
print("model_name:", cfg["model_name"])
print("target_phrases:", cfg["target_phrases"])
print("n_samples:", cfg["n_samples"])
print("✅ YAML 構文OK")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# オプション: Colab 無料枠での動作確認用に小さい設定に書き換え
# 本番学習には実行しないでください
# ──────────────────────────────────────────────────────────────────────────
QUICK_TEST = False  # True にすると小さい設定で動作確認

if QUICK_TEST:
    import yaml, copy
    with open(CONFIG) as f:
        quick_cfg = yaml.safe_load(f)
    quick_cfg["n_samples"] = 48
    quick_cfg["n_samples_val"] = 16
    quick_cfg["n_background_samples"] = 40
    quick_cfg["n_background_samples_val"] = 8
    quick_cfg["steps"] = 500
    quick_cfg["model"]["model_type"] = "dnn"
    quick_cfg["model"]["model_size"] = "tiny"
    quick_cfg["voxcpm_tts"]["voice_design_prompts"] = [
        "A young adult woman, clear mid-pitch voice, moderate pace",
        "A young adult man, warm baritone, steady pace",
    ]
    quick_cfg["voxcpm_tts"]["cfg_values"] = [2.0]
    quick_cfg["voxcpm_tts"]["inference_timesteps_list"] = [10]
    QUICK_CONFIG = "/content/sensei_ja_quick.yaml"
    with open(QUICK_CONFIG, "w") as f:
        yaml.dump(quick_cfg, f, allow_unicode=True)
    CONFIG = QUICK_CONFIG
    print(f"✅ クイックテスト設定を {CONFIG} に書き込みました")
    print(f"   n_samples={quick_cfg['n_samples']}, steps={quick_cfg['steps']}")

## 4.5 Google Drive 連携（マウント・復元・自動バックアップ）

無料 Colab はコンピューティングユニットが切れるとセッションが終了し、`/content` 以下のデータは消えます。**VoxCPM2 モデルや生成済みクリップは Google Drive の専用フォルダに自動保存・復元**するようにして、何度セッションが切れても途中から再開できるようにします。

- バックアップ先: `/content/drive/MyDrive/livekit-wakeword-ja/`
- 他の Drive データとは別フォルダなので、既存ファイルと混ざりません


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, glob

DRIVE_DIR = "/content/drive/MyDrive/livekit-wakeword-ja"
os.makedirs(DRIVE_DIR, exist_ok=True)


def backup_to_drive():
    """output/sensei_ja と data/voxcpm を Drive にバックアップし、件数を表示する。"""
    if os.path.exists("data/voxcpm") and not os.path.exists(f"{DRIVE_DIR}/voxcpm"):
        print("VoxCPM2 を Drive にバックアップ中... (初回のみ、数分かかります)")
        shutil.copytree("data/voxcpm", f"{DRIVE_DIR}/voxcpm")

    if os.path.exists("output/sensei_ja"):
        dest = f"{DRIVE_DIR}/sensei_ja"
        if os.path.exists(dest):
            shutil.rmtree(dest)
        shutil.copytree("output/sensei_ja", dest)
        n_pos = len(glob.glob(f"{dest}/positive_train/*.wav"))
        n_neg = len(glob.glob(f"{dest}/negative_train/*.wav"))
        n_bg = len(glob.glob(f"{dest}/background_train/*.wav"))
        print(f"\u2705 Drive バックアップ完了 — positive: {n_pos}, negative: {n_neg}, background: {n_bg}")


def restore_from_drive():
    """前回セッションのデータを Drive から復元する。"""
    restored = False
    if os.path.exists(f"{DRIVE_DIR}/voxcpm") and not os.path.exists("data/voxcpm"):
        os.makedirs("data", exist_ok=True)
        shutil.copytree(f"{DRIVE_DIR}/voxcpm", "data/voxcpm")
        print("\u2705 VoxCPM2 モデルを Drive から復元しました")
        restored = True
    if os.path.exists(f"{DRIVE_DIR}/sensei_ja") and not os.path.exists("output/sensei_ja"):
        os.makedirs("output", exist_ok=True)
        shutil.copytree(f"{DRIVE_DIR}/sensei_ja", "output/sensei_ja")
        print("\u2705 生成済みデータを Drive から復元しました")
        restored = True
    if not restored:
        print("\u2139\ufe0f Drive に復元対象のデータはありません（初回実行、またはバックアップ未作成）")


restore_from_drive()

## 5. セットアップ（モデル・データのダウンロード）

> **⚠️ 注意**: VoxCPM2 モデルは数 GB あります。初回ダウンロードに 10〜30 分かかる場合があります。

In [ ]:
!livekit-wakeword setup --config {CONFIG}

## 6. 合成データ生成（小分け実行 + 自動バックアップ）

VoxCPM2 を使って「先生」「せんせい」のポジティブクリップ、`custom_negative_phrases` のアドバーサリアルネガティブクリップ、背景ノイズクリップを生成します。

**無料 Colab はコンピューティングユニットが途中で切れることが多いため、目標件数まで一度に生成せず、少しずつ件数を増やしながら生成 → 都度 Drive にバックアップする方式にしています。** セッションがどこで切れても、直前のバックアップ時点から再開できます。

- `CHUNK_SIZE` 件ずつ `n_samples` を増やして `generate` を実行（既存クリップはスキップされ、差分のみ作られます）
- 1 チャンク完了ごとに `backup_to_drive()` を呼んで Drive に保存
- 途中でランタイムが切れた場合は、このセルをそのまま再実行すれば直前のチャンクから続行されます

> **⚠️ TODO**: VoxCPM2 の日本語発音品質はエンドツーエンドで未検証です。下のセルで生成後に再生して確認してください。


In [ ]:
import yaml

TARGET_N_SAMPLES = 25000   # examples/sensei_ja.yaml の本番値
CHUNK_SIZE = 500           # 1回のセッションで確実に終わる目安の件数

with open(CONFIG) as f:
    base_cfg = yaml.safe_load(f)

n = min(CHUNK_SIZE, TARGET_N_SAMPLES)
while True:
    cfg = dict(base_cfg)
    cfg["n_samples"] = n
    cfg["n_background_samples"] = 0  # 背景ノイズは次のセルで別途生成
    chunk_path = "examples/sensei_ja_chunk.yaml"
    with open(chunk_path, "w") as f:
        yaml.dump(cfg, f, allow_unicode=True)

    print(f"\n=== positive/negative: n_samples={n} まで生成中 ===")
    !livekit-wakeword generate {chunk_path}
    backup_to_drive()

    if n >= TARGET_N_SAMPLES:
        break
    n = min(n + CHUNK_SIZE, TARGET_N_SAMPLES)

print("\n\u2705 ポジティブ / ネガティブ生成完了")

In [ ]:
TARGET_N_BACKGROUND = 2000
BG_CHUNK_SIZE = 200

n_bg = min(BG_CHUNK_SIZE, TARGET_N_BACKGROUND)
while True:
    cfg = dict(base_cfg)
    cfg["n_samples"] = TARGET_N_SAMPLES        # 既に完了済みなのでスキップされる
    cfg["n_background_samples"] = n_bg
    chunk_path = "examples/sensei_ja_chunk.yaml"
    with open(chunk_path, "w") as f:
        yaml.dump(cfg, f, allow_unicode=True)

    print(f"\n=== background: n_background_samples={n_bg} まで生成中 ===")
    !livekit-wakeword generate {chunk_path}
    backup_to_drive()

    if n_bg >= TARGET_N_BACKGROUND:
        break
    n_bg = min(n_bg + BG_CHUNK_SIZE, TARGET_N_BACKGROUND)

print("\n\u2705 背景ノイズ生成完了")

In [ ]:
# 生成されたクリップの確認
import glob, os
output_dir = "output/sensei_ja"
for subdir in ["positive_train", "negative_train", "background_train"]:
    path = os.path.join(output_dir, subdir)
    files = glob.glob(os.path.join(path, "*.wav"))
    print(f"{subdir}: {len(files)} クリップ")

In [ ]:
# ポジティブクリップを再生して品質確認
import glob, random
from IPython.display import Audio, display

positive_files = sorted(glob.glob("output/sensei_ja/positive_train/*.wav"))
if positive_files:
    for f in random.sample(positive_files, min(3, len(positive_files))):
        print(f"再生: {f}")
        display(Audio(f))
else:
    print("ポジティブクリップが見つかりません。generate ステップを確認してください。")

## 7. オーグメンテーション

バックグラウンドノイズ・RIR（残響）・EQ などを適用して多様な学習データを生成します。

In [ ]:
!livekit-wakeword augment {CONFIG}

## 8. 学習

Conv-Attention 分類器を学習します。

- デフォルト設定: `medium` サイズ、`100,000` ステップ
- Colab T4 で概算 **1〜4 時間**
- チェックポイントは `output/sensei_ja/` に保存されます

> **Colab セッションリセット後の再開**: このセルを再実行すると自動的に途中から再開します。

In [ ]:
!livekit-wakeword train {CONFIG}

## 9. ONNX エクスポート

In [ ]:
!livekit-wakeword export {CONFIG}

In [ ]:
import os
onnx_path = "output/sensei_ja/sensei_ja.onnx"
if os.path.exists(onnx_path):
    size_mb = os.path.getsize(onnx_path) / 1024 / 1024
    print(f"✅ ONNX エクスポート成功: {onnx_path} ({size_mb:.1f} MB)")
else:
    print(f"❌ {onnx_path} が見つかりません。export ステップを確認してください。")

## 10. 評価

検証セットで Recall / FPPH / AUT を計算します。

| 指標 | 目標 |
|---|---|
| FPPH | < 0.2 |
| Recall | ≥ 80% |
| AUT | < 0.1 |

In [ ]:
!livekit-wakeword eval {CONFIG}

## 11. モデルのダウンロード

In [ ]:
from google.colab import files
import os

onnx_path = "output/sensei_ja/sensei_ja.onnx"
if os.path.exists(onnx_path):
    files.download(onnx_path)
    print("✅ ダウンロードを開始しました")
else:
    print(f"❌ {onnx_path} が見つかりません。export ステップを先に実行してください。")

## 12. トラブルシューティング

---

### Colab ランタイムがリセットされた

「4.5 Google Drive 連携」セルの `backup_to_drive()` / `restore_from_drive()` を使ってください。生成ステップはチャンクごとに自動で Drive へバックアップされるので、ランタイムが切れても直前のチャンクまでのデータが残っています。

再接続したら、上から順にセルを再実行するだけで構いません（`restore_from_drive()` が前回分を復元し、チャンク生成ループが続きから自動で再開します）。

学習（`train`）のチェックポイントも Drive に保存しておきたい場合は、学習後に手動で以下を実行してください:

```python
backup_to_drive()
```

---

### GPU が割り当てられない

「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU** を選択してください。GPU なしでは VoxCPM2 の合成が非常に遅くなります（10〜100 倍）。

---

### VoxCPM2 のセットアップ・インポートが失敗する

```
ModuleNotFoundError: No module named 'voxcpm'
```

→ `pip install -e ".[train,voxcpm]"` を再実行してください。

VoxCPM2 は PyTorch 2.5 以上を推奨します。バージョン確認:

```python
import torch; print(torch.__version__)
```

---

### 音声ファイルが見つからない

`generate` の前に `setup` が完了しているか確認してください。セットアップが途中で失敗した場合は `data/backgrounds/` や `data/rirs/` が空になっています:

```bash
ls data/backgrounds/
ls data/rirs/
```

---

### 学習が遅すぎる

- `n_samples` を 2,000〜5,000 に減らす（QUICK_TEST オプションを参照）
- `steps` を 10,000〜30,000 に減らす
- `model_size` を `small` または `tiny` に変更する
- GPU ランタイムが正しく設定されているか確認する

---

### VoxCPM2 が日本語を正しく発音しない

> **⚠️ TODO**: VoxCPM2 の日本語合成はエンドツーエンドで未検証です。

セル 6 の音声再生で確認してください。発音が不自然な場合:
- `target_phrases` に「せんせい」（ひらがな）のみを使う（漢字より音声的な読みに近い場合がある）
- `voice_design_prompts` の日本語話者向けプロンプト（日本語で書かれた5種類）のみに絞る
- 問題が継続する場合は [livekit-wakeword Issues](https://github.com/livekit/livekit-wakeword/issues) に報告してください